# 3. Adaptive Controller

This notebook explains how to simulate the adaptive controller.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using LinkedMasses

## Parameter Vector and Regressor

The drawback of the feedback-linearizing controller from the previous notebook is that it requires accurate knowledge of the parameters of the system. In practice, parameters such as damping and ocean currents are not known exactly.

Let us assume that only the length $L$ is known. Then, the control input of the feedback-linearizing controller can be written in the following form

$$ u = Y \zeta $$

where $Y$ is a regressor matrix of the system states and known parameters, and $\zeta$ is a vector of unknown parameters. The parameter vector can be constructed from system parameters using the function `parameter_vector`:

In [ ]:
@doc parameter_vector

## Adaptive Controller

Let $\hat{\zeta}$ be an estimate of the unknown parameters. Then, using the following control law

$$ u = Y \hat{\zeta} $$

$$ \dot{\hat{\zeta}} = -\gamma Y^\top \tilde{v}_\varepsilon $$

where $\gamma$ is a positive adaptation gain, guarantees convergence.

The augmented state vector (system state + path parameter + parameter estimate) can be represented using the `AdaptiveSimulationState` struct:

In [ ]:
@doc AdaptiveSimulationState

The parameters of the adaptive controller are contained in the `AdaptiveLinearizingController` struct:

In [ ]:
@doc AdaptiveLinearizingController

## Example

### System Parameters

We take the linked-masses model from Notebook 1:

In [ ]:
V_c = [-0.2, 0.0]

m0 = 200.0
m = 40.0
c0 = 250.0
c = 50.0
L = 10.0

params = LinkedMassesParameters(L, m0, m, c0, c)

### Parameter Estimate

To find a reasonable initial value of $\hat{\zeta}$, we create an approximate model. We set the mass of the payload to double and the damping coefficient to half its true value:

In [ ]:
k_err = 1.5
params_estimate = LinkedMassesParameters(L, m0, m, k_err*c0, c/k_err)

Next, we use the `parameter_vector` function to find an initial value of $\hat{\zeta}$. We set the ocean current velocity to zero:

In [ ]:
ε = 0.7
ζ_hat0 = parameter_vector(params_estimate, ε, zeros(2))

## Adaptive Controller Parameters

We take the path, LOS guidance parameters, and velocity gain from the previous notebook. The adaptation gain is set to 1:

In [ ]:
# Path
R = 10.0
ϕ0 = -π / 2
p_origin = [2.0, 12.0]
path_circle(s) = p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]

# Guidance parameters
U = 1.0
Δ = 5.0
los = LOSParameters(U, Δ)

# Controller gains
k_v = 0.1
γ = 5.0

ctrl = AdaptiveLinearizingController(params, los, path_circle, k_v, γ, ε, V_c)

In [ ]:
using DifferentialEquations
T_stop = 300

# Initial state: both masses at rest, first mass at origin, second mass east of it
p0_init = [0.0, 0.0]
θ_init = -π / 2
v0_init = [0.0, 0.0]
ω_init = 0.0
s_init = 0.0
init_state = AdaptiveSimulationState(p0_init, θ_init, v0_init, ω_init, s_init, ζ_hat0)

prob = ODEProblem(closed_loop_ode!, pack_state(init_state), (0.0, T_stop), ctrl)
sol = solve(prob, Tsit5())

### Closed-Loop Errors

Use `get_errors` to calculate the path-following and velocity errors:

In [ ]:
@doc get_errors

In [ ]:
## Plot the error states and export to CSV
folder = joinpath(@__DIR__, "csv")
mkpath(folder)

# Interpolate to get error states at uniform time intervals
t_vals = range(0, stop=T_stop, length=151)
sol_interp = sol(t_vals)
e_x, e_y, v_x, v_y = get_errors(sol_interp, ctrl)
theta_dot = [u[6] for u in sol_interp]

# Payload position
p = hcat([payload_position(u, params) for u in sol_interp]...)'

# Control point
p0 = hcat([u[1:2] for u in sol_interp]...)'
p_ε = (1 - ε) * p0 + ε * p

# Path
s = [u[7] for u in sol_interp]
p_path = hcat(path_circle.(s)...)'

using CSV
using DataFrames
df = DataFrame(time = t_vals,
               e_x = e_x,
               e_y = e_y,
               v_x = v_x,
               v_y = v_y,
               x0 = [u[1] for u in sol_interp],
               y0 = [u[2] for u in sol_interp],
               x = p[:, 1],
               y = p[:, 2],
               x_eps = p_ε[:, 1],
               y_eps = p_ε[:, 2],
               x_path = p_path[:, 1],
               y_path = p_path[:, 2],
               theta_dot = theta_dot)
CSV.write(joinpath(folder, "adaptive_controller.csv"), df)

using Plots
using LaTeXStrings
fig1 = plot(t_vals, e_x, label=L"$e_x$ (m)", xlabel="Time (s)", ylabel="Error (m)", legend=:topright, title="Path Following Errors")
plot!(t_vals, e_y, label=L"$e_y$ (m)")
fig2 = plot(t_vals, v_x, label=L"$v_x$ (m/s)", xlabel="Time (s)", ylabel="Error (m/s)", legend=:topright, title="Velocity Errors")
plot!(t_vals, v_y, label=L"$v_y$ (m/s)")
fig3 = plot(t_vals, theta_dot, label=L"\dot{\theta} (rad/s)", xlabel="Time (s)", ylabel="Angular Velocity (rad/s)", legend=:topright, title="Angular Velocity")
plot(fig1, fig2, fig3, layout=@layout([[a b]; c]), size=(800,600))

In [ ]:
## Plot the trajectory and export to CSV
# Path
fig_traj = plot(p_path[:, 2], p_path[:, 1], title="Trajectory", xlabel="East (m)", ylabel="North (m)", aspect_ratio=1, label="Path", lw=2, lc=:black)
# Control point trajectory
plot!(fig_traj, p_ε[:, 2], p_ε[:, 1], label="Control Point", lw=1.5)
# Towed system at times t = [0, 45, 75, 100] s
t_snapshots = [0.0, 45.0, 75.0, 100.0]
x0_snapshots = Float64[]
y0_snapshots = Float64[]
θ_snapshots = Float64[]
x_path_snapshots = Float64[]
y_path_snapshots = Float64[]
for t_snap in t_snapshots
    u_snap = sol(t_snap)
    push!(x0_snapshots, u_snap[1])
    push!(y0_snapshots, u_snap[2])
    push!(θ_snapshots, u_snap[3])
    push!(x_path_snapshots, path_circle(u_snap[7])[1])
    push!(y_path_snapshots, path_circle(u_snap[7])[2])
    plot_linked_masses!(fig_traj, u_snap, params; label="t = $(t_snap) s", lw=1.5, marker=:o)
    scatter!(fig_traj, [path_circle(u_snap[7])[2]], [path_circle(u_snap[7])[1]], label="", ms=4, mc=:black, marker=:square)
end

df_snap = DataFrame(time = t_snapshots,
                    x0 = x0_snapshots,
                    y0 = y0_snapshots,
                    theta = θ_snapshots,
                    x_path = x_path_snapshots,
                    y_path = y_path_snapshots)
CSV.write(joinpath(folder, "adaptive_controller_snapshots.csv"), df_snap)

fig_traj